# Apache Arrow Flight

![Arrow Logo](images/arrow.png)


# The Scenario

![City Bikes](images/citybike.jpg)

> Photo by <a href="https://unsplash.com/@jacegrandinetti?utm_source=unsplash&utm_medium=referral&utm_content=creditCopyText">Jace & Afsoon</a> on <a href="https://unsplash.com/photos/assorted-color-bicycles-park-beside-blue-rails-near-river-VEXIwDcY1gw?utm_source=unsplash&utm_medium=referral&utm_content=creditCopyText">Unsplash</a>

We have our CityBike API, and we need to fetch 1,000,000 records

We need to fetch and process all the records for our analysis.

## With REST API

We have access to a REST API, looks something like this, classic JSON REST API

In [22]:
import httpx2
import polars as pl

# REST_API_URL = "https://arrow-flight-rest.fly.dev"
REST_API_URL = "http://rest:8000"

In [23]:
# Wake up the remote server
req = httpx2.get(f"{REST_API_URL}/health")
req.json()

{'status': 'ok', 'db': 'ok'}

In [24]:
req = httpx2.get(f"{REST_API_URL}/data/rides/all?num_rows=100", headers={"Authorization": "Bearer pydata_amsterdam"})
pl.from_records(req.json())

ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
str,str,str,str,str,str,str,str,f64,f64,f64,f64,str
"""85744AF35D7F2DF5""","""electric_bike""","""2026-01-02T05:36:24.539000""","""2026-01-02T05:42:21.153000""","""W 42 St & 8 Ave""","""6602.05""","""E 58 St & Madison Ave""","""6839.04""",40.75757,-73.990985,40.763026,-73.972095,"""member"""
"""9D18958E5788880B""","""electric_bike""","""2026-01-02T15:15:11.915000""","""2026-01-02T15:18:40.462000""","""Division St & Bowery""","""5311.08""","""Clinton St & Grand St""","""5303.06_""",40.71419,-73.99673,40.715738,-73.98699,"""member"""
"""B050891B7B009EE5""","""electric_bike""","""2026-01-12T10:12:51.453000""","""2026-01-12T10:16:55.731000""","""Broadway & 31 St""","""6789.08""","""35 Ave & 37 St""","""6563.12""",40.76194,-73.92513,40.755733,-73.923661,"""member"""
"""0B6D7938C4EF1668""","""electric_bike""","""2026-01-01T01:03:29.712000""","""2026-01-01T01:05:37.341000""","""34 St & 35 Ave""","""6605.08""","""35 St & Broadway""","""6750.16""",40.756933,-73.926223,40.760339,-73.922243,"""member"""
"""95415F60C7120CC3""","""electric_bike""","""2026-01-03T19:55:51.636000""","""2026-01-03T20:14:57.964000""","""E 6 St & Ave B""","""5584.04""","""W 55 St & 6 Ave""","""6809.09""",40.724537,-73.981854,40.763189,-73.978434,"""member"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""34A3DEC468BAA97E""","""electric_bike""","""2026-01-05T09:10:26.063000""","""2026-01-05T09:12:49.360000""","""University Pl & E 14 St""","""5905.14""","""Washington Pl & Greene St""","""5755.15""",40.734814,-73.992085,40.729806,-73.995464,"""member"""
"""A28F5F71AEF77BB9""","""electric_bike""","""2026-01-05T17:50:06.706000""","""2026-01-05T18:01:17.873000""","""E 55 St & 2 Ave""","""6650.07""","""E 89 St & York Ave""","""7204.08""",40.757973,-73.966033,40.777945,-73.946041,"""member"""
"""85EFC5E0885C81ED""","""classic_bike""","""2026-01-11T20:03:00.136000""","""2026-01-11T20:10:36.555000""","""Harlem River Dr & W 155 St""","""8085.13""","""Frederick Douglass Blvd & W 13…","""7876.07""",40.830476,-73.939929,40.819006,-73.944769,"""member"""


In [25]:
%%timeit -r 1
req = httpx2.get("https://arrow-flight-rest.fly.dev/data/rides/all", headers={"Authorization": "Bearer pydata_amsterdam"})
pl.from_records(req.json())

ReadTimeout: The read operation timed out

## With Arrow Flight

Let's try that again, but with an Arrow Flight server instead

In [26]:
from pyarrow import flight

#FLIGHT_SERVER_URL = "grpc+tls://arrow-flight-server.fly.dev:443"
FLIGHT_SERVER_URL = "grpc://server:7001"

In [27]:
client = flight.connect(FLIGHT_SERVER_URL)
# Wake up the remote server
client.wait_for_available(5)

In [28]:
%%timeit -r 1
info = client.get_flight_info(flight.FlightDescriptor.for_path("rides"))
data = client.do_get(info.endpoints[0].ticket)
pl.from_arrow(data.read_all())

182 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


## Why Arrow Flight?

Arrow is a foundational technology in the Data Engineering space. It powers all your favourite tools
 from Spark to Polars, Duckdb and Snowflake. As they say, it's a standard

![XKDC Standards](images/Standards.png)

Why this particular standard matters, is that it solves the problem of interprocess communication
 between different languages and frameworks.

## Process interop

Take the scenario of the Spark UDF in the old days:

![ipc](images/spark_udf.png)

Without a standard for memory layout for the data, we need to introduce glue code between the JVM
Spark Memory and the Pandas Numpy memory layout, usually having to copy the data back and forth.


When we introduce Apache Arrow, both runtimes can share the same memory layout, and we can avoid
the need for any glue code or copying.
![ipc](images/spark_arrow.png)


This is true for any library which uses Arrow, like Duckdb, Pandas and Polars.

![I made this](images/i_made_this.jpg)


## Arrow as the data interchange format
Having Arrow as an universal data interchange format allows libraries to delegate responsibility for their memory layout and compute to 
Arrow and instead focus on their value-adding layer. 

We've seen this before - compilers came along, and gave us all these different optimizations and now we no longer inline statements or unravel loops. 

LLVM and JVM are both examples of the power of separating the layers of a program, so that we can focus on the value-adding part.

So Arrow is great - but what is Arrow Flight then?
